In [0]:
import pandas as pd
from pyspark.sql import functions as F

# Step 1: Read with Pandas (handles messy CSV better!)
pdf = pd.read_csv(
    '/Workspace/swiggy_analytics_project/data/zomato.csv',
    on_bad_lines='skip',  # skip corrupted rows
    encoding='utf-8'
)

print(f"Pandas rows: {len(pdf)}")
print(f"Pandas columns: {len(pdf.columns)}")

# Step 2: Convert to Spark DataFrame
df_bronze = spark.createDataFrame(pdf)

print("Converted to Spark successfully!")

Pandas rows: 51717
Pandas columns: 17
Converted to Spark successfully!


In [0]:
# Step 3: Clean the data
from pyspark.sql import functions as F

# Drop columns we don't need
df_clean = df_bronze.drop(
    "url", "phone", "menu_item", 
    "reviews_list", "dish_liked"
)

# Clean rate column (remove "/5", handle "NEW")
df_clean = df_clean.withColumn(
    "rate",
    F.when(
        F.col("rate").isin("NEW", "-", "null", ""),
        None
    ).otherwise(
        F.regexp_replace(F.col("rate"), "/5", "").cast("float")
    )
)

# Clean approx_cost (remove everything except numbers)
df_clean = df_clean.withColumn(
    "approx_cost(for two people)",
    F.regexp_replace(
        F.col("approx_cost(for two people)"),
        "[^0-9]", ""
    ).cast("integer")
)

# Clean votes to integer
df_clean = df_clean.withColumn(
    "votes", F.col("votes").cast("integer")
)

# Rename messy column names
df_clean = df_clean \
    .withColumnRenamed("approx_cost(for two people)", "approx_cost") \
    .withColumnRenamed("listed_in(type)", "listed_type") \
    .withColumnRenamed("listed_in(city)", "listed_city")

# Drop nulls in important columns
df_clean = df_clean.dropna(
    subset=["name", "location", "rate", "votes"]
)

print(f"Clean Rows: {df_clean.count()}")
print(f"Clean Columns: {len(df_clean.columns)}")
df_clean.printSchema()

Clean Rows: 41665
Clean Columns: 12
root
 |-- address: string (nullable = true)
 |-- name: string (nullable = true)
 |-- online_order: string (nullable = true)
 |-- book_table: string (nullable = true)
 |-- rate: float (nullable = true)
 |-- votes: integer (nullable = true)
 |-- location: string (nullable = true)
 |-- rest_type: string (nullable = true)
 |-- cuisines: string (nullable = true)
 |-- approx_cost: integer (nullable = true)
 |-- listed_type: string (nullable = true)
 |-- listed_city: string (nullable = true)



In [0]:
# Save to Volumes path (writable in Free Edition)
df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("zomato_silver")

print("Silver Delta Table saved successfully! ")

Silver Delta Table saved successfully! 


In [0]:
# Verify Delta table exists and read it back
df_silver = spark.read.table("zomato_silver")

print(f"Rows in Delta table: {df_silver.count()}")
print(f"Columns: {len(df_silver.columns)}")

# Show sample data
display(df_silver.limit(5))

Rows in Delta table: 41665
Columns: 12


address,name,online_order,book_table,rate,votes,location,rest_type,cuisines,approx_cost,listed_type,listed_city
"96, 100 Feet Ring Road, Vysya Bank Colony, 1st Stage, BTM, Bangalore",Magix's Parattha Roll,Yes,No,3.8,184,BTM,Quick Bites,"Fast Food, North Indian, Chinese, Mughlai, Rolls",400,Delivery,Jayanagar
"2B Block, Total Mall, Hosur Road, Madiwala",KFC,Yes,No,4.0,351,BTM,Quick Bites,"Burger, Fast Food",400,Delivery,Jayanagar
"406, 2nd Cross, Vidyapeeta Layout, Banashankri, Bangalore",Kababs N Biryani,Yes,No,3.4,23,Banashankari,Quick Bites,South Indian,200,Delivery,Jayanagar
"17, Gandhi Bazaar Main Road, Basavanagudi, Bangalore",Roti Ghar,Yes,No,3.9,753,Basavanagudi,Casual Dining,"North Indian, Chinese, South Indian",500,Delivery,Jayanagar
"Shop 1, 100 Feet Ring Road, 1st Phase, Stage 2, BTM, Bangalore",Shaadi Ki Biryani,Yes,No,3.1,31,BTM,"Takeaway, Delivery","Biryani, North Indian, Mughlai",400,Delivery,Jayanagar
